# 13 - GLOBAL-test dgn LOCAL normalization (Paper 4, poin reviewer P0-2)

**Masalah yang diangkat reviewer:** evaluasi GLOBAL-test di nb03/nb12 memakai
**pooled z-score** (statistik gabungan train) untuk monitoring, padahal protokol
training menegakkan **z-score LOKAL per-klien**. Ini bisa membuat distribusi input
saat testing tidak identik dgn saat training -> menyentuh *construct validity*.

**Yang diuji:** ulangi evaluasi GLOBAL-test dgn **normalisasi lokal konsisten
deployment**: tiap sampel dinormalisasi dgn statistik TRAIN klien ASALNYA
(CIC-test pakai z-score CIC-train; UNSW-test pakai z-score UNSW-train), lalu
prediksi digabung. Bandingkan dgn angka pooled-z-score di naskah.

**Harapan (untuk naskah):** jika hasil ~sama -> tulis 'results remain stable under
deployment-consistent local normalization' (memperkuat paper). Jika beda -> lapor
apa adanya & pakai angka local-norm sebagai yang utama.

**Cakupan:** FedAvg naif + FedPer (dua yang pakai GLOBAL-test). MLP 9->128->64->1
identik nb03/nb12. Output: `fed_out/local_norm_check.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + MLP (identik nb12)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, balanced_accuracy_score
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEED=42; LAYERS=[9,128,64,1]
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def init_w(seed=SEED):
    r=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1); a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; return _sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs=LOCAL_EPOCHS,lr=LR,batch=256,seed=SEED):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2)=forward(W,Xb); g=(out-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]): W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(ws,sizes,idxs=None):
    tot=float(sum(sizes)); out=[w.copy() for w in ws[0]]
    for i in (range(len(ws[0])) if idxs is None else idxs):
        out[i]=sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws)))
    return out
def zfit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zap(X,mu,sd): return (X-mu)/sd
print('setup ok')

## 1. Muat partisi

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    yg_te=np.concatenate([yc_te,yu_te])
    print('loaded. GLOBAL-test size', len(yg_te))
else: print('(jalankan nb01 dulu)')

## 2. Helper: dua skema evaluasi GLOBAL-test

- **pooled** (seperti naskah lama): z-score gabungan train utk semua sampel.
- **local** (deployment-consistent): CIC-test pakai z CIC-train; UNSW-test pakai
  z UNSW-train; prediksi digabung. Inilah yang diminta reviewer.

In [ ]:
def mcc(y,yp): return round(matthews_corrcoef(y,yp),4)
def pred(W,X,mu,sd): out,_=forward(W,zap(X,mu,sd)); return (out>=0.5).astype(int)
def eval_global_pooled(Wcic_head_or_global, mu_g,sd_g):
    # satu model global (FedAvg): pooled z-score utk CIC-test+UNSW-test
    yp=np.concatenate([pred(Wcic_head_or_global,Xc_te,mu_g,sd_g),
                       pred(Wcic_head_or_global,Xu_te,mu_g,sd_g)])
    return mcc(yg_te,yp)
def eval_global_local(Wglobal, zc_cic, zc_unsw):
    # satu model global tapi tiap domain pakai z-score TRAIN domainnya
    yp=np.concatenate([pred(Wglobal,Xc_te,*zc_cic),
                       pred(Wglobal,Xu_te,*zc_unsw)])
    return mcc(yg_te,yp)
print('helper siap')

## 3. FedAvg naif: GLOBAL-test pooled vs local z-score

In [ ]:
res=[]
if ok:
    clients={'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}
    zc={cid:zfit(X) for cid,(X,y) in clients.items()}
    Xz={cid:zap(clients[cid][0],*zc[cid]) for cid in clients}
    sizes=[len(yc_tr),len(yu_tr)]
    mu_g,sd_g=zfit(np.vstack([Xc_tr,Xu_tr]))
    Wg=init_w(SEED)
    for t in range(ROUNDS):
        W1=train_local(Wg,Xz['cic'],yc_tr); W2=train_local(Wg,Xz['unsw'],yu_tr)
        Wg=fedavg([W1,W2],sizes)
    g_pooled=eval_global_pooled(Wg,mu_g,sd_g)
    g_local =eval_global_local(Wg,zc['cic'],zc['unsw'])
    res.append({'method':'FedAvg','GLOBAL_pooled_z':g_pooled,'GLOBAL_local_z':g_local})
    print('FedAvg  pooled',g_pooled,'| local',g_local)
print('=== SEL 3 (FedAvg) SELESAI ===')

## 4. FedPer: GLOBAL-test (personalized head) sudah pakai z-score lokal

FedPer memang sudah memakai z-score lokal per-domain (head klien asal). Kita
hitung ulang eksplisit utk konfirmasi + sajikan sebagai pembanding.

In [ ]:
if ok:
    BACK=[0,1,2,3]
    Wg=init_w(SEED); Wloc={c:[w.copy() for w in Wg] for c in clients}
    for t in range(ROUNDS):
        upd={}
        for cid in clients:
            Ws=[Wg[i] if i in BACK else Wloc[cid][i] for i in range(6)]
            Wk=train_local(Ws,Xz[cid],clients[cid][1]); upd[cid]=Wk
            Wloc[cid]=[Wk[i].copy() for i in range(6)]
        Wg=fedavg([upd['cic'],upd['unsw']],sizes,idxs=BACK)
    # GLOBAL-test: tiap domain pakai head domainnya + z-score lokal domainnya
    def pred_dom(cid,Xte):
        Wp=[Wg[i] if i in BACK else Wloc[cid][i] for i in range(6)]
        return pred(Wp,Xte,*zc[cid])
    yp=np.concatenate([pred_dom('cic',Xc_te),pred_dom('unsw',Xu_te)])
    gper_local=mcc(yg_te,yp)
    res.append({'method':'FedPer','GLOBAL_pooled_z':'(n/a: personalized)','GLOBAL_local_z':gper_local})
    print('FedPer  local',gper_local)
print('=== SEL 4 (FedPer) SELESAI ===')

## 5. Simpan + interpretasi

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'local_norm_check.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    print('\nBANDING: GLOBAL_pooled_z vs GLOBAL_local_z.')
    print('Jika FedAvg pooled~local -> hasil STABIL thd skema normalisasi (perkuat paper).')
    print('Pembanding naskah: FedAvg pooled -0.026 ; FedPer local 0.731.')
else: print('(lewati: jalankan nb01)')
print('=== SEL 5 (hasil) SELESAI ===')

## 6. Unggah ke S3 (results/local_norm)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1'); PREFIX='unsw-far/federated/results/local_norm'
try:
    import boto3; s3=boto3.client('s3',region_name=REGION)
    p=os.path.join(OUTDIR,'local_norm_check.csv')
    if os.path.exists(p): s3.upload_file(p,S3_BUCKET,f'{PREFIX}/local_norm_check.csv'); print('ok ->',PREFIX)
except Exception as e: print('(upload gagal:',e,')')
print('=== SEL 6 (unggah) SELESAI ===')

## 7. Catatan untuk naskah (poin reviewer P0-2)

- Reviewer: evaluasi GLOBAL-test harus konsisten deployment (z-score LOKAL),
  bukan pooled. nb ini menyediakan kedua angka utk FedAvg + FedPer local.
- Jika `GLOBAL_pooled_z` ~ `GLOBAL_local_z` (FedAvg) -> ganti Threats-to-Validity
  jadi klaim positif: 'results remain stable under deployment-consistent local
  normalization'. Masukkan angka NYATA.
- FedPer memang sudah local-z-score (head per-domain); angka 0.731 di naskah
  sudah deployment-consistent -> konfirmasi di sini.